In [0]:
# Silver Payments

from pyspark.sql import functions as F

CATALOG = "insurance_lakehouse"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
QUARANTINE_SCHEMA = "quarantine"

bronze_table = f"{CATALOG}.{BRONZE_SCHEMA}.bronze_payments"
silver_table = f"{CATALOG}.{SILVER_SCHEMA}.silver_payments"
quarantine_table = f"{CATALOG}.{QUARANTINE_SCHEMA}.quarantine_invalid_payments"

valid_payment_status = ["paid", "pending", "rejected"]
valid_payment_method = ["SEPA", "bank_transfer", "card"]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{QUARANTINE_SCHEMA}")

payments_bronze = spark.table(bronze_table)

silver_claims = (
    spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_claims")
    .select(
        F.col("claim_id").alias("valid_claim_id"),
        F.col("claim_date").alias("parent_claim_date")
    )
    .dropDuplicates(["valid_claim_id"])
)

payments_prepared = (
    payments_bronze
    .withColumn("payment_id", F.trim(F.col("payment_id")))
    .withColumn("claim_id", F.trim(F.col("claim_id")))
    .withColumn("payment_date", F.to_date(F.col("payment_date")))
    .withColumn("payment_amount", F.col("payment_amount").cast("double"))
    .withColumn("payment_status", F.lower(F.trim(F.col("payment_status"))))
    .withColumn(
        "payment_method",
        F.when(F.upper(F.trim(F.col("payment_method"))) == "SEPA", F.lit("SEPA"))
         .otherwise(F.lower(F.trim(F.col("payment_method"))))
    )
    .withColumn("created_at", F.to_timestamp(F.col("created_at")))
)

payments_joined = (
    payments_prepared
    .join(silver_claims, F.col("claim_id") == F.col("valid_claim_id"), "left")
    .withColumn("claim_exists", F.col("valid_claim_id").isNotNull())
    .drop("valid_claim_id")
)

duplicate_payment_ids = (
    payments_joined
    .groupBy("payment_id")
    .count()
    .filter(F.col("payment_id").isNotNull() & (F.col("count") > 1))
    .select("payment_id")
    .withColumn("is_duplicate_payment_id", F.lit(True))
)

payments_checked = (
    payments_joined
    .join(duplicate_payment_ids, on="payment_id", how="left")
    .withColumn(
        "is_duplicate_payment_id",
        F.coalesce(F.col("is_duplicate_payment_id"), F.lit(False))
    )
)

invalid_condition = (
    F.col("payment_id").isNull()
    | F.col("is_duplicate_payment_id")
    | F.col("claim_id").isNull()
    | (~F.col("claim_exists"))
    | F.col("payment_date").isNull()
    | (F.col("payment_date") < F.col("parent_claim_date"))
    | F.col("payment_amount").isNull()
    | (F.col("payment_amount") < 0)
    | F.col("payment_status").isNull()
    | (~F.col("payment_status").isin(valid_payment_status))
    | F.col("payment_method").isNull()
    | (~F.col("payment_method").isin(valid_payment_method))
)

error_reason = (
    F.when(F.col("payment_id").isNull(), F.lit("missing_payment_id"))
    .when(F.col("is_duplicate_payment_id"), F.lit("duplicate_payment_id"))
    .when(F.col("claim_id").isNull(), F.lit("missing_claim_id"))
    .when(~F.col("claim_exists"), F.lit("missing_claim_reference"))
    .when(F.col("payment_date").isNull(), F.lit("missing_payment_date"))
    .when(F.col("payment_date") < F.col("parent_claim_date"), F.lit("payment_before_claim_date"))
    .when(F.col("payment_amount").isNull(), F.lit("missing_payment_amount"))
    .when(F.col("payment_amount") < 0, F.lit("invalid_payment_amount"))
    .when(F.col("payment_status").isNull(), F.lit("missing_payment_status"))
    .when(~F.col("payment_status").isin(valid_payment_status), F.lit("invalid_payment_status"))
    .when(F.col("payment_method").isNull(), F.lit("missing_payment_method"))
    .when(~F.col("payment_method").isin(valid_payment_method), F.lit("invalid_payment_method"))
    .otherwise(F.lit("unknown_payment_quality_issue"))
)

invalid_payments = (
    payments_checked
    .filter(invalid_condition)
    .withColumn("record_id", F.col("payment_id").cast("string"))
    .withColumn("source_table", F.lit("bronze_payments"))
    .withColumn("error_reason", error_reason)
    .withColumn("error_severity", F.lit("HIGH"))
    .withColumn("quarantine_timestamp", F.current_timestamp())
    .withColumn(
        "original_record_json",
        F.to_json(F.struct(*[F.col(c) for c in payments_prepared.columns]))
    )
    .select(
        "record_id",
        "source_table",
        "error_reason",
        "error_severity",
        "quarantine_timestamp",
        "source_file_name",
        "ingest_run_id",
        "original_record_json"
    )
)

valid_payments = (
    payments_checked
    .filter(~invalid_condition)
    .drop("claim_exists", "parent_claim_date", "is_duplicate_payment_id")
    .dropDuplicates(["payment_id"])
)

valid_payments.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)

invalid_payments.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(quarantine_table)

print("Bronze payments:", payments_bronze.count())
print("Silver payments:", spark.table(silver_table).count())
print("Quarantine payments:", spark.table(quarantine_table).count())